### Semantic Chunking
- SemanticChunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.

- It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
# Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# step 1: Split into sentences
sentences = [s.strip() for s in text.split('\n') if s.strip()]

# step 2: Embed each sentence
embeddings = model.encode(sentences)

# step 3: Initialize parameters
threshold = 0.7
chunks = []
current_chunk = [sentences[0]]

# step 4: semantic grouping based on threshold
for i in range(1, len(sentences)):
    sim = cosine_similarity([embeddings[i]], [embeddings[0]])[0][0]
    if sim >= threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

# Append the last chunk
chunks.append(" ".join(current_chunk))


print("Semantic Chunks:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Semantic Chunks:
Chunk 1: LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
Chunk 2: You can create chains, agents, memory, and retrievers.
Chunk 3: The Eiffel Tower is located in Paris.
Chunk 4: France is a popular tourist destination.


# RAG Pipeline

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_classic.schema import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.schema.runnable import RunnableLambda, RunnableMap
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")


In [6]:
# Custom Semantic chunker with Threshold

class ThresholdSemanticChunker:
    def __init__(self, model_name = "all-MiniLM-L6-v2", threshold=0.7):
        self.threshold = threshold
        self.model = SentenceTransformer(model_name)

    def split(self, text):
        # step 1: Split into sentences
        sentences = [s.strip() for s in text.split('\n') if s.strip()]

        # step 2: Embed each sentence
        embeddings = model.encode(sentences)

        # step 3: Initialize parameters
        chunks = []
        current_chunk = [sentences[0]]

        # step 4: semantic grouping based on threshold
        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i]], [embeddings[0]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]

        # Append the last chunk
        chunks.append(" ".join(current_chunk))

        return chunks
    
    def split_documents(self, documents):
        chunks = []
        for doc in documents:
            doc_chunks = self.split(doc.page_content)
            for chunk in doc_chunks:
                chunks.append(Document(page_content=chunk, metadata=doc.metadata))
        return chunks

In [7]:
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""
doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [9]:
# chunking
chunker = ThresholdSemanticChunker(threshold=0.7)
chunks = chunker.split_documents([doc])
chunks

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [10]:
# vector store
embeddings = OpenAIEmbeddings(
    api_key=api_key,
    base_url=base_url
)
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever()

In [11]:
# Prompt Template

# --- 5. Prompt Template ---
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [12]:
## LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.4,
    api_key=api_key,
    base_url=base_url
)

# LCEL Chain With Retrieval
rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"]
         }
    )
    | prompt
    | llm
    | StrOutputParser()
)

query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

result

'LangChain is used for building applications with large language models (LLMs). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'

# Semantic Chunker With LangChain

In [13]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

In [14]:
loader = TextLoader("langchain_intro.txt")
documents = loader.load()

embeddings = OpenAIEmbeddings()

chunker = SemanticChunker(embeddings)

chunks = chunker.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk.page_content}")

Chunk 0: LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
Chunk 1: You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.
